# Classification Project

In [98]:
import numpy as np
import pandas as pd
from sklearn import preprocessing

In [99]:
df = pd.read_csv('./heart.csv')
print(df['output'].value_counts())
print(df.shape)
df.head()

1    165
0    138
Name: output, dtype: int64
(303, 14)


,age,sex,cp,trtbps,chol,fbs,restecg,thalachh,exng,oldpeak,slp,caa,thall,output
0,63,1,3,145,233,1,0,150,0,2.3,0,0,1,1
1,37,1,2,130,250,0,1,187,0,3.5,0,0,2,1
2,41,0,1,130,204,0,0,172,0,1.4,2,0,2,1
3,56,1,1,120,236,0,1,178,0,0.8,2,0,2,1
4,57,0,0,120,354,0,1,163,1,0.6,2,0,2,1


In [100]:
X = df.drop(['output'], axis=1).values
X[0:3]

array([[ 63. ,   1. ,   3. , 145. , 233. ,   1. ,   0. , 150. ,   0. ,
          2.3,   0. ,   0. ,   1. ],
       [ 37. ,   1. ,   2. , 130. , 250. ,   0. ,   1. , 187. ,   0. ,
          3.5,   0. ,   0. ,   2. ],
       [ 41. ,   0. ,   1. , 130. , 204. ,   0. ,   0. , 172. ,   0. ,
          1.4,   2. ,   0. ,   2. ]])

In [101]:
y = df['output'].values
y[0:3]

array([1, 1, 1])

### because the data set is not large we first try KNN

In [102]:
scaler = preprocessing.StandardScaler().fit(X)
X = scaler.transform(X.astype(float))
X[0:3]

array([[ 0.9521966 ,  0.68100522,  1.97312292,  0.76395577, -0.25633371,
         2.394438  , -1.00583187,  0.01544279, -0.69663055,  1.08733806,
        -2.27457861, -0.71442887, -2.14887271],
       [-1.91531289,  0.68100522,  1.00257707, -0.09273778,  0.07219949,
        -0.41763453,  0.89896224,  1.63347147, -0.69663055,  2.12257273,
        -2.27457861, -0.71442887, -0.51292188],
       [-1.47415758, -1.46841752,  0.03203122, -0.09273778, -0.81677269,
        -0.41763453, -1.00583187,  0.97751389, -0.69663055,  0.31091206,
         0.97635214, -0.71442887, -0.51292188]])

In [103]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=4)

In [104]:
from sklearn.neighbors import KNeighborsClassifier
clf = KNeighborsClassifier(n_neighbors=4)
clf.fit(X_train, y_train)

KNeighborsClassifier(n_neighbors=4)

In [105]:
y_hat = clf.predict(X_test)
print(y_hat[:5])
print(y[:5])

[0 0 1 1 0]
[1 1 1 1 1]


In [106]:
from sklearn import metrics
print(metrics.accuracy_score(y_test, y_hat))
print(metrics.accuracy_score(y_train, clf.predict(X_train)))

0.8032786885245902
0.859504132231405


In [107]:
print(metrics.f1_score(y_test, y_hat))

0.8181818181818182


In [108]:
Ks = 50
mean_acc = np.zeros(Ks-1)
for i in range(1, Ks):
    clf = KNeighborsClassifier(n_neighbors = i)
    clf.fit(X_train, y_train)
    y_hat = clf.predict(X_test)
    mean_acc[i-1] = metrics.f1_score(y_hat, y_test)
print(mean_acc.max(), "with k equal to", mean_acc.argmax()+1)

0.9090909090909091 with k equal to 37


### with KNN it seems that we have an acceptable accuracy

### now we check SVM

In [109]:
X = df.drop(['output'], axis=1).values
y = df['output'].values

In [110]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=4)

In [111]:
from sklearn import svm
from sklearn.metrics import f1_score
kernels = ['rbf', 'linear', 'poly', 'sigmoid']
accuracies = []
for kernel in kernels:
    clf = svm.SVC(kernel=kernel)
    clf.fit(X_train, y_train)
    y_hat = clf.predict(X_test)
    accuracy = f1_score(y_test, y_hat, average='weighted')
    accuracies.append(accuracy)

In [112]:
for kernel, acc in zip(kernels, accuracies):
    print(kernel, acc)

rbf 0.6511395441823271
linear 0.9177578060193311
poly 0.7148771083197313
sigmoid 0.4380598276153456


### as we see the linear kernel gives the best prediction so the data set is linear

### here we check the Logistic Regression as well to see the f1_score and compare

In [113]:
X = df.drop(['output'], axis=1).values
y = df['output'].values

In [114]:
scaler = preprocessing.StandardScaler().fit(X)
X = scaler.transform(X)

In [115]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=4)

In [116]:
from sklearn.linear_model import LogisticRegression
LR = LogisticRegression(C=0.01, solver='lbfgs').fit(X_train, y_train)

In [117]:
y_hat = LR.predict(X_test)
print(y_hat)
print(y_test)

[1 0 1 1 1 1 0 1 1 1 1 1 1 0 1 0 1 1 0 1 1 1 0 1 1 0 0 1 1 0 0 1 1 1 1 1 1
 1 1 0 0 0 0 1 1 1 0 0 0 1 0 1 0 1 1 1 1 1 0 0 1]
[1 0 1 1 0 0 0 1 1 1 1 1 1 0 1 0 1 1 0 0 1 1 0 1 1 0 0 1 1 0 0 1 1 1 0 1 1
 1 0 0 0 0 0 1 1 1 0 0 0 1 0 1 1 1 1 1 1 1 0 0 1]


In [118]:
y_hat_prob = LR.predict_proba(X_test)
y_hat_prob[:6]

array([[0.35294385, 0.64705615],
       [0.70005228, 0.29994772],
       [0.37869319, 0.62130681],
       [0.17615514, 0.82384486],
       [0.37723072, 0.62276928],
       [0.47625541, 0.52374459]])

In [119]:
print(f1_score(y_hat, y_test))

0.9210526315789473


### I have tested the options and the best somehow is c=0.01 and solver='lbfgs'

### now we test the last one the decision tree algorithm

In [120]:
X = df.drop(['output'], axis=1).values
y = df['output'].values

In [121]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=4)

In [122]:
from sklearn.tree import DecisionTreeClassifier
clf = DecisionTreeClassifier(criterion='entropy', max_depth=4) 

In [123]:
clf.fit(X_train, y_train)

DecisionTreeClassifier(criterion='entropy', max_depth=4)

In [124]:
y_hat = clf.predict(X_test)
print(y_test[0:5])
print(y_hat[0:5])

[1 0 1 1 0]
[0 0 1 1 1]


In [125]:
from sklearn.metrics import f1_score
print(f1_score(y_hat, y_test))

0.8799999999999999


## Conclusion

according to the f1_scores, we notice that all 4 algorithms work pretty good (all scores greater than 85) but the best results are from logistic regression and svm algorithms.